# Аналитика онлайн-школы

Учебная работа Т-Академии: объединение выгрузок платежей и расчёт продуктовых метрик по каналам привлечения за июль–сентябрь 2025 года. Основа — сохранённое решение домашнего задания. Для публикации сокращён текст и изменены пути к данным.

## Данные и метрики

`users` — пользователи, `lessons` — уроки, три месячные выгрузки — платежи.

- Выручка: сумма `amount` по успешным платежам пользователей, присутствующих в `users`.
- MAU: уникальные пользователи с уроком в месяце.
- Платящие: пользователи хотя бы с одним успешным платежом.
- ARPU = выручка / MAU; ARPPU = выручка / платящие.
- Доля возвратов: количество платежей со статусом `refunded` / все платежи.

MAU и платящие считаются по разным событиям. `payers / mau` здесь — отношение двух баз, а не конверсия одной когорты. При нулевом знаменателе метрика остаётся `NaN`.

In [1]:
import pandas as pd
import numpy as np

from IPython.display import display

## 1. Объединение платежей

В сентябре отличаются названия колонок и формат суммы. Сначала сравниваем схему, затем приводим её к общему виду и объединяем выгрузки.

In [2]:
p07 = pd.read_excel('data/payments_2025_07.xlsx')
p08 = pd.read_excel('data/payments_2025_08.xlsx')
p09 = pd.read_excel('data/payments_2025_09.xlsx')

for month, table in [('Июль', p07), ('Август', p08), ('Сентябрь', p09)]:
    print(month)
    print(table.shape)
    print(table.columns)
    print(table.dtypes)
    print()

pay_raw = pd.concat([p07, p08, p09], ignore_index=True)
print('Количество колонок после склейки без исправлений:', pay_raw.shape[1])
display(pay_raw.head())
display(pay_raw.tail())
print(pay_raw.isna().sum())

p09 = p09.rename(columns={'client_id': 'user_id', 'payment_sum': 'amount'})
p09['amount'] = p09['amount'].astype(str).str.replace(r'\s+', '', regex=True).astype(int)

payments = pd.concat([p07, p08, p09], ignore_index=True)
payments['user_id'] = payments['user_id'].astype(int)

print('Число строк совпадает:', len(payments) == len(p07) + len(p08) + len(p09))
print('Тип amount:', payments['amount'].dtype)
print('Всего платежей:', len(payments))
print('Уникальных пользователей:', payments['user_id'].nunique())

promo_pct = p08['promo_code'].fillna('').astype(str).str.strip().ne('').mean() * 100
print('Доля августовских платежей с промокодом:', round(promo_pct, 2), '%')
display(payments.head())

Июль
(170, 7)
Index(['payment_id', 'user_id', 'payment_date', 'product_type', 'plan',
       'amount', 'status'],
      dtype='object')
payment_id               int64
user_id                  int64
payment_date    datetime64[ns]
product_type            object
plan                    object
amount                   int64
status                  object
dtype: object

Август
(340, 8)
Index(['payment_id', 'user_id', 'payment_date', 'product_type', 'plan',
       'amount', 'status', 'promo_code'],
      dtype='object')
payment_id               int64
user_id                  int64
payment_date    datetime64[ns]
product_type            object
plan                    object
amount                   int64
status                  object
promo_code              object
dtype: object

Сентябрь
(373, 7)
Index(['payment_id', 'client_id', 'payment_date', 'product_type', 'plan',
       'payment_sum', 'status'],
      dtype='object')
payment_id               int64
client_id                int64
payment_

,payment_id,user_id,payment_date,product_type,plan,amount,status,promo_code,client_id,payment_sum
0,9100001,300484.0,2025-07-01 11:11:00,course,NaN,7900.0,success,NaN,NaN,NaN
1,9100002,301331.0,2025-07-01 18:35:00,course,NaN,4900.0,success,NaN,NaN,NaN
2,9100003,303230.0,2025-07-02 08:30:00,subscription,Basic,990.0,success,NaN,NaN,NaN
3,9100004,302122.0,2025-07-03 15:51:00,course,NaN,12900.0,success,NaN,NaN,NaN
4,9100005,999003.0,2025-07-03 16:00:00,subscription,Basic,1.0,success,NaN,NaN,NaN


,payment_id,user_id,payment_date,product_type,plan,amount,status,promo_code,client_id,payment_sum
878,9100879,NaN,2025-09-30 11:27:00,subscription,Pro,NaN,success,NaN,300816.0,1 990
879,9100880,NaN,2025-09-30 11:44:00,subscription,Basic,NaN,success,NaN,300659.0,990
880,9100881,NaN,2025-09-30 15:01:00,subscription,Basic,NaN,success,NaN,300058.0,990
881,9100882,NaN,2025-09-30 15:01:00,subscription,Basic,NaN,success,NaN,301083.0,990
882,9100883,NaN,2025-09-30 19:27:00,subscription,Basic,NaN,success,NaN,301649.0,990


,payment_id,user_id,payment_date,product_type,plan,amount,status,promo_code
0,9100001,300484,2025-07-01 11:11:00,course,NaN,7900,success,NaN
1,9100002,301331,2025-07-01 18:35:00,course,NaN,4900,success,NaN
2,9100003,303230,2025-07-02 08:30:00,subscription,Basic,990,success,NaN
3,9100004,302122,2025-07-03 15:51:00,course,NaN,12900,success,NaN
4,9100005,999003,2025-07-03 16:00:00,subscription,Basic,1,success,NaN


## 2. Проверка соединений

Сравниваем результаты разных типов соединения. Платежи без пользователя выделяем отдельно и исключаем из расчёта по правилу учебного кейса. Их происхождение по этим данным не установлено.

In [3]:
users = pd.read_excel('data/school_q3_2025.xlsx', sheet_name='users')
lessons = pd.read_excel('data/school_q3_2025.xlsx', sheet_name='lessons')

print('users:', len(users), users['user_id'].nunique())
print('payments:', len(payments), payments['user_id'].nunique())

joins = {}
for how in ['inner', 'left', 'right', 'outer']:
    joins[how] = payments.merge(users, on='user_id', how=how)
    print(how, len(joins[how]))

pay_no_user = payments[~payments['user_id'].isin(users['user_id'])]
user_no_pay = users[~users['user_id'].isin(payments['user_id'])]
print('left по сравнению с inner добавляет платежи без пользователя:', len(pay_no_user))
print('right включает пользователей без платежей:', len(user_no_pay))
print('outer по сравнению с right добавляет платежи без пользователя:', len(pay_no_user))
display(pay_no_user)

pay_ok = payments[
    payments['user_id'].isin(users['user_id']) & (payments['status'] == 'success')
].copy()

print('Выручка за Q3:', pay_ok['amount'].sum(), 'руб.')
print('Платящих за Q3:', pay_ok['user_id'].nunique())

users: 4000 4000
payments: 883 630
inner 871
left 883
right 4253
outer 4265
left по сравнению с inner добавляет платежи без пользователя: 12
right включает пользователей без платежей: 3382
outer по сравнению с right добавляет платежи без пользователя: 12
Выручка за Q3: 2381160 руб.
Платящих за Q3: 587


,payment_id,user_id,payment_date,product_type,plan,amount,status,promo_code
4,9100005,999003,2025-07-03 16:00:00,subscription,Basic,1,success,NaN
41,9100042,999004,2025-07-10 16:00:00,subscription,Basic,1,success,NaN
224,9100225,999005,2025-08-05 14:00:00,subscription,Basic,1,success,AUG15
256,9100257,999010,2025-08-08 14:00:00,subscription,Basic,1,success,NaN
283,9100284,999006,2025-08-11 16:00:00,subscription,Basic,1,success,NaN
366,9100367,999009,2025-08-19 18:00:00,subscription,Basic,1,success,NaN
401,9100402,999008,2025-08-23 13:00:00,subscription,Basic,1,success,NaN
524,9100525,999002,2025-09-02 09:00:00,subscription,Basic,1,success,NaN
707,9100708,999007,2025-09-16 16:00:00,subscription,Basic,1,success,NaN
721,9100722,999012,2025-09-18 11:00:00,subscription,Basic,1,success,NaN


## 3. Метрики по месяцам и каналам

In [4]:
lesson_user = lessons.merge(users, left_on='student_id', right_on='user_id', how='inner')
lesson_user['month'] = pd.to_datetime(lesson_user['lesson_start']).dt.to_period('M')
lesson_user = lesson_user[
    lesson_user['month'].between(pd.Period('2025-07'), pd.Period('2025-09'))
]

mau_month = lesson_user.groupby(['month', 'acquisition_channel']).agg(
    mau=('student_id', 'nunique')
).reset_index()

pay_user = pay_ok.merge(users, on='user_id', how='inner')
pay_user['month'] = pd.to_datetime(
    pay_user['payment_date'].astype(str).str[:7], format='%Y-%m'
).dt.to_period('M')

pay_month = pay_user.groupby(['month', 'acquisition_channel']).agg(
    payers=('user_id', 'nunique'),
    revenue=('amount', 'sum')
).reset_index()

metrics = mau_month.merge(
    pay_month, on=['month', 'acquisition_channel'], how='outer'
)
metrics[['mau', 'payers', 'revenue']] = metrics[['mau', 'payers', 'revenue']].fillna(0).astype(int)
metrics['arpu'] = metrics['revenue'] / metrics['mau'].replace(0, np.nan)
metrics['arppu'] = metrics['revenue'] / metrics['payers'].replace(0, np.nan)
metrics['paying_share_%'] = metrics['payers'] / metrics['mau'].replace(0, np.nan) * 100
metrics = metrics.sort_values(['month', 'acquisition_channel']).reset_index(drop=True)
display(metrics)

september = metrics[metrics['month'] == pd.Period('2025-09')]
arpu_top = september.loc[september['arpu'] == september['arpu'].max(), 'acquisition_channel'].tolist()
arppu_top = september.loc[september['arppu'] == september['arppu'].max(), 'acquisition_channel'].tolist()
print('Максимальный ARPU в сентябре:', arpu_top)
print('Максимальный ARPPU в сентябре:', arppu_top)
print('Лидеры совпадают:', set(arpu_top) == set(arppu_top))

Максимальный ARPU в сентябре: ['Organic']
Максимальный ARPPU в сентябре: ['Referral']
Лидеры совпадают: False


,month,acquisition_channel,mau,payers,revenue,arpu,arppu,paying_share_%
0,2025-07,Ads,176,17,36240,205.909091,2131.764706,9.659091
1,2025-07,Blogger,132,35,147960,1120.909091,4227.428571,26.515152
2,2025-07,Organic,309,74,259090,838.478964,3501.216216,23.948220
3,2025-07,Partner,66,15,35170,532.878788,2344.666667,22.727273
4,2025-07,Referral,104,7,27160,261.153846,3880.000000,6.730769
5,2025-08,Ads,337,44,112230,333.026706,2550.681818,13.056380
6,2025-08,Blogger,252,65,178150,706.944444,2740.769231,25.793651
7,2025-08,Organic,613,150,437140,713.115824,2914.266667,24.469821
8,2025-08,Partner,129,24,55480,430.077519,2311.666667,18.604651
9,2025-08,Referral,238,23,81230,341.302521,3531.739130,9.663866


## 4. Возвраты по каналам

In [5]:
pay_real = payments[payments['user_id'].isin(users['user_id'])].copy()
pay_real = pay_real.merge(users, on='user_id', how='inner')
pay_real['is_refunded'] = np.where(pay_real['status'] == 'refunded', 1, 0)
pay_real['refunded_amount'] = np.where(
    pay_real['status'] == 'refunded', pay_real['amount'], 0
)

refunds = pay_real.groupby('acquisition_channel').agg(
    payments=('payment_id', 'size'),
    refunded_amount=('refunded_amount', 'sum'),
    refund_rate=('is_refunded', 'mean')
).reset_index()
refunds['refund_rate_%'] = refunds['refund_rate'] * 100
refunds = refunds[['acquisition_channel', 'payments', 'refunded_amount', 'refund_rate_%']]
refunds = refunds.sort_values('refund_rate_%', ascending=False).reset_index(drop=True)
display(refunds)

refund_top = refunds.loc[
    refunds['refund_rate_%'] == refunds['refund_rate_%'].max(), 'acquisition_channel'
].tolist()
print('Наибольшая доля возвратов:', refund_top)

Наибольшая доля возвратов: ['Blogger']


,acquisition_channel,payments,refunded_amount,refund_rate_%
0,Blogger,195,50960,13.846154
1,Ads,120,37640,7.500000
2,Partner,75,11950,6.666667
3,Referral,69,14370,5.797101
4,Organic,412,59380,4.126214


## Выводы

В сентябре максимальный ARPU у Organic, а максимальный ARPPU у Referral: это разные вопросы, поскольку меняется знаменатель метрики. По доле возвратов выделяется Blogger. Гипотеза для следующей проверки — несоответствие рекламных ожиданий содержанию обучения.

Различия между каналами описательные. Для решения о бюджете нужны расходы, сравнимые когорты и причины возвратов. Рассчитанная выручка не равна прибыли.